In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [6]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Animereccomdationmodel/clean_anime_data.csv')

Mounted at /content/drive


In [7]:
print(df.shape)

(3801, 23)


In [8]:
print(df.columns.tolist())

['title', 'english_title', 'romaji_title', 'genres', 'rating_anilist', 'rating_mal', 'episodes', 'status', 'format', 'season', 'year', 'synopsis', 'studio', 'source_material', 'popularity_anilist', 'mal_members', 'mal_rank', 'cover_image_url', 'start_date', 'end_date', 'duration_per_ep', 'age_rating', 'mal_id']


In [9]:
df['features'] = (
    df['genres'].astype(str) + ' ' +
    df['studio'].astype(str) + ' ' +
    df['format'].astype(str) + ' ' +
    df['source_material'].astype(str) + ' ' +
    df['season'].astype(str) + ' ' +
    df['age_rating'].astype(str)
)

print(df['features'].head(10))

0    Action | Drama | Fantasy | Mystery WIT STUDIO ...
1    Action | Adventure | Drama | Fantasy | Superna...
2    Action | Drama | Supernatural MAPPA TV MANGA F...
3    Mystery | Psychological | Supernatural | Thril...
4    Action | Adventure | Comedy bones TV MANGA SPR...
5    Action | Adventure | Fantasy MADHOUSE TV MANGA...
6    Action | Comedy | Sci-Fi | Supernatural MADHOU...
7    Action | Drama | Horror | Mystery | Psychologi...
8    Action | Adventure | Comedy | Drama | Fantasy ...
9    Action | Drama | Fantasy | Mystery WIT STUDIO ...
Name: features, dtype: object


In [10]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

In [11]:
stop_words='english'

In [12]:
max_features=5000

In [13]:
tfidf_matrix = tfidf.fit_transform(df['features'])

In [14]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 44519 stored elements and shape (3801, 376)>

In [15]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [16]:
(tfidf_matrix, tfidf_matrix)

(<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 44519 stored elements and shape (3801, 376)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 44519 stored elements and shape (3801, 376)>)

In [17]:
cosine_sim

array([[1.        , 0.48452444, 0.41853516, ..., 0.28188564, 0.05166843,
        0.23733099],
       [0.48452444, 1.        , 0.5067322 , ..., 0.24704956, 0.05243664,
        0.26059123],
       [0.41853516, 0.5067322 , 1.        , ..., 0.21175762, 0.05849811,
        0.18872225],
       ...,
       [0.28188564, 0.24704956, 0.21175762, ..., 1.        , 0.23918946,
        0.32400429],
       [0.05166843, 0.05243664, 0.05849811, ..., 0.23918946, 1.        ,
        0.12686488],
       [0.23733099, 0.26059123, 0.18872225, ..., 0.32400429, 0.12686488,
        1.        ]])

In [18]:
print("Similarity matrix shape:", cosine_sim.shape)

Similarity matrix shape: (3801, 3801)


In [19]:
def get_recommendations(title, n=10):

    idx = df[df['title'].str.lower() == title.lower()].index
    if len(idx) == 0:
        idx = df[df['english_title'].str.lower() == title.lower()].index
    if len(idx) == 0:
        return f"Anime '{title}' not found."

    idx = idx[0]
    input_title = df.iloc[idx]['title'].lower()


    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)


    filtered = []
    for i, score in sim_scores:
        if i == idx:
            continue  # Skip itself
        rec_title = df.iloc[i]['title'].lower()

        if input_title in rec_title or rec_title in input_title:
            continue
        filtered.append((i, score))
        if len(filtered) == n:
            break

    anime_indices = [i[0] for i in filtered]
    results = df[['title', 'english_title', 'genres', 'rating_mal', 'cover_image_url']].iloc[anime_indices]
    results['similarity_score'] = [round(i[1], 3) for i in filtered]

    return results

    idx = df[df['title'].str.lower() == title.lower()].index
    if len(idx) == 0:
        idx = df[df['english_title'].str.lower() == title.lower()].index
    if len(idx) == 0:
        return f"Anime '{title}' not found in dataset."
    idx = idx[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1] # Exclude itself
    anime_indices = [i[0] for i in sim_scores]
    results = df[['title', 'english_title', 'genres', 'rating_mal', 'cover_image_url']].iloc[anime_indices]
    results['similarity_score'] = [round(i[1], 3) for i in sim_scores]
    return results

In [25]:
get_recommendations('Attack on Titan')

,title,english_title,genres,rating_mal,cover_image_url,similarity_score
119,Seraph of the End: Vampire Reign,Seraph of the End: Vampire Reign,Action | Drama | Fantasy | Mystery | Supernatural,7.49,https://s4.anilist.co/file/anilistcdn/media/an...,0.966
283,Seraph of the End: Battle in Nagoya,Seraph of the End: Battle in Nagoya,Action | Drama | Fantasy | Supernatural,7.61,https://s4.anilist.co/file/anilistcdn/media/an...,0.847
1804,Seraph of the End: Kyuuketsuki Shahal,Seraph of the End: Kyuuketsuki Shahal,Action | Drama | Fantasy | Supernatural,7.18,https://s4.anilist.co/file/anilistcdn/media/an...,0.839
40,Vinland Saga,Vinland Saga,Action | Adventure | Drama,8.78,https://s4.anilist.co/file/anilistcdn/media/an...,0.819
298,Kabaneri of the Iron Fortress,Kabaneri of the Iron Fortress,Action | Drama | Fantasy | Horror | Supernatural,7.29,https://s4.anilist.co/file/anilistcdn/media/an...,0.784
1482,Kabaneri of the Iron Fortress: The Battle of U...,Kabaneri of the Iron Fortress: The Battle of U...,Action | Drama | Fantasy | Horror | Supernatural,7.70,https://s4.anilist.co/file/anilistcdn/media/an...,0.740
1177,Suicide Squad ISEKAI,Suicide Squad ISEKAI,Action | Adventure | Fantasy | Sci-Fi,6.36,https://s4.anilist.co/file/anilistcdn/media/an...,0.711
1365,Ranking of Kings: The Treasure Chest of Courage,Ranking of Kings: The Treasure Chest of Courage,Action | Adventure | Drama | Fantasy,7.42,https://s4.anilist.co/file/anilistcdn/media/an...,0.707
233,Great Pretender,Great Pretender,Action | Comedy | Drama | Thriller,8.19,https://s4.anilist.co/file/anilistcdn/media/an...,0.698
3367,Kabaneri of the Iron Fortress: Light That Gathers,Kabaneri of the Iron Fortress: Light That Gathers,Action | Drama | Fantasy | Horror | Supernatural,7.59,https://s4.anilist.co/file/anilistcdn/media/an...,0.688


In [24]:
get_recommendations('Noragami')

,title,english_title,genres,rating_mal,cover_image_url,similarity_score
88,Soul Eater,Soul Eater,Action | Adventure | Comedy | Fantasy | Supern...,7.86,https://s4.anilist.co/file/anilistcdn/media/an...,0.878
1864,Mob Psycho 100 REIGEN The Miraculous Unknown P...,Mob Psycho 100 REIGEN The Miraculous Unknown P...,Action | Comedy | Supernatural,7.41,https://s4.anilist.co/file/anilistcdn/media/an...,0.853
4,My Hero Academia,My Hero Academia,Action | Adventure | Comedy,7.83,https://s4.anilist.co/file/anilistcdn/media/an...,0.852
16,My Hero Academia Season 2,My Hero Academia Season 2,Action | Adventure | Comedy,8.05,https://s4.anilist.co/file/anilistcdn/media/an...,0.852
1617,Soul Eater Not!,Soul Eater Not!,Action | Comedy | Fantasy | Supernatural,5.87,https://s4.anilist.co/file/anilistcdn/media/an...,0.826
183,My Hero Academia Season 6,My Hero Academia Season 6,Action | Adventure,8.22,https://s4.anilist.co/file/anilistcdn/media/an...,0.824
361,My Hero Academia Season 7,My Hero Academia Season 7,Action | Adventure,8.14,https://s4.anilist.co/file/anilistcdn/media/an...,0.824
27,My Hero Academia Season 3,My Hero Academia Season 3,Action | Adventure | Comedy | Drama,7.98,https://s4.anilist.co/file/anilistcdn/media/an...,0.821
1815,Boku no Hero Academia: Training of the Dead,Boku no Hero Academia: Training of the Dead,Action | Adventure | Comedy | Supernatural,7.21,https://s4.anilist.co/file/anilistcdn/media/an...,0.817
3218,Boku no Hero Academia THE MOVIE: World Heroes'...,Boku no Hero Academia THE MOVIE: World Heroes'...,Action | Comedy,7.43,https://s4.anilist.co/file/anilistcdn/media/an...,0.794


In [23]:
get_recommendations('Haikyu')

"Anime 'Haikyu' not found."